# MPS311/439: Week 5 Lab Solution - Linear & Quadratic Discriminant Analysis

**Duration**: 50 minutes + 30 minutes (MPS439)  
**Objective**: Learn to use LDA and QDA for classification, understand when to use each method, and compare with logistic regression

---

This notebook contains **complete solutions** with detailed explanations of:
- **WHY** we're doing each step
- **WHAT** the code does
- **HOW** to interpret the results

# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load Iris dataset
iris = load_iris()
X_full = iris.data
y_full = iris.target
y = y_full  # Keep both names for consistency

# For visualization, we'll use only 2 features: sepal length and sepal width
# This makes it easy to plot decision boundaries
X = X_full[:, [0, 1]]  # Columns 0 and 1
feature_names = [iris.feature_names[0], iris.feature_names[1]]

# Split into train and test (70/30 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Number of classes: {len(iris.target_names)}")
print(f"Class names: {iris.target_names}")
print(f"Features used: {feature_names}")
print("\nSetup complete! ✓")

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load Iris dataset
iris = load_iris()
X_full = iris.data
y_full = iris.target

# For visualization, we'll use only 2 features: sepal length and sepal width
# This makes it easy to plot decision boundaries
X = X_full[:, [0, 1]]  # Columns 0 and 1
feature_names = [iris.feature_names[0], iris.feature_names[1]]

# Split into train and test (70/30 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Number of classes: {len(iris.target_names)}")
print(f"Class names: {iris.target_names}")
print(f"Features used: {feature_names}")
print("\nSetup complete! ✓")

**Expected Output**:
```
Training set size: 105
Test set size: 45
Number of classes: 3
Class names: ['setosa' 'versicolor' 'virginica']
Features used: ['sepal length (cm)', 'sepal width (cm)']

Setup complete! ✓
```

# Create LDA model
# SOLUTION: store_covariance=True
lda_model = LinearDiscriminantAnalysis(store_covariance=True)

# Fit the model on training data
# SOLUTION: X_train, y_train
lda_model.fit(X_train, y_train)

# Make predictions on test data
# SOLUTION: X_test
y_pred_lda = lda_model.predict(X_test)

# Calculate accuracy
accuracy_lda = accuracy_score(y_test, y_pred_lda)
print(f"LDA Test Accuracy: {accuracy_lda:.3f}")

### Task 1.1: Fit LDA Model

**WHY**: We want to train an LDA model to learn the distribution of each class.

**WHAT**: 
- `store_covariance=True`: This allows us to access the learned covariance matrix later
- `fit()`: Learns the class means and shared covariance from training data
- `predict()`: Classifies new points based on which class distribution they're most likely from

**HOW**: LDA internally computes discriminant functions for each class and picks the class with the highest score.

In [ ]:
# Create LDA model
# SOLUTION: n_components=2
lda_model = LinearDiscriminantAnalysis(store_covariance=True)

# Fit the model on training data
# SOLUTION: X_train, y_train
lda_model.fit(X_train, y_train)

# Make predictions on test data
# SOLUTION: X_test
y_pred_lda = lda_model.predict(X_test)

# Calculate accuracy
accuracy_lda = accuracy_score(y_test, y_pred_lda)
print(f"LDA Test Accuracy: {accuracy_lda:.3f}")

**Expected Output**: Accuracy around 0.82-0.84 (may vary slightly due to random split)

**Interpretation**: This accuracy tells us how well LDA separates the three iris species using just sepal measurements.

### Task 1.2: Access Learned Parameters

**WHY**: Understanding what LDA learned helps us interpret the model.

**WHAT**: 
- `means_`: The center (mean vector) of each class
- `covariance_`: The shared covariance matrix (same for all classes!)
- `priors_`: The proportion of each class in training data

**HOW**: These parameters define the Gaussian distribution for each class.

In [ ]:
# Access the learned class means (mu_k for each class k)
class_means = lda_model.means_
print("Class Means (centers of each class):")
print(class_means)
print()

# Access the shared covariance matrix (Sigma)
shared_cov = lda_model.covariance_
print("Shared Covariance Matrix (same for all classes):")
print(shared_cov)
print()

# Access class priors (how common each class is)
priors = lda_model.priors_
print("Class Priors:")
for i, class_name in enumerate(iris.target_names):
    print(f"  {class_name}: {priors[i]:.3f}")

**Answers to Questions**:
1. **How many class means?** 3 (one for each iris species)
2. **Shared across how many classes?** All 3 classes share the same covariance matrix
3. **Most common class?** Should be roughly equal (~0.333 each) since Iris is balanced

**Key Insight**: The single covariance matrix is what makes LDA "linear" - all classes have the same shape!

### Task 1.3: Visualize LDA Decision Boundaries

**WHY**: Visualization helps us see the linear boundaries LDA creates.

**WHAT**: 
- We create a grid of points covering the feature space
- Predict the class for each grid point
- Color regions by predicted class

**HOW**: The `contourf` function creates colored regions, and we overlay training points to see how well boundaries separate classes.

In [ ]:
# Create a mesh to plot decision boundaries
h = 0.02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predict class for each point in the mesh
Z = lda_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors='k', cmap='viridis')
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title('LDA Decision Boundaries')
plt.show()

**Answer**: The decision boundaries are **straight lines**. This is because LDA assumes equal covariance, which leads to linear discriminant functions.

**Observation**: You should see 3 regions (one for each class) separated by straight lines. Some classes may overlap slightly where the model makes mistakes.

---

## Part 2: Quadratic Discriminant Analysis (QDA) (10 minutes)

### Background
**WHY QDA?** QDA relaxes LDA's equal covariance assumption. Each class can have its **own covariance matrix**.

**KEY DIFFERENCE**: Different covariances → different shapes → **curved boundaries** (quadratic)

**TRADE-OFF**: QDA is more flexible but needs more data to estimate all those covariances reliably.

### Task 2.1: Fit QDA Model

**WHAT**: The API is identical to LDA, just a different class.

**HOW**: QDA estimates a separate covariance matrix for each class, then uses these in the discriminant function.

In [ ]:
# Create QDA model
qda_model = QuadraticDiscriminantAnalysis(store_covariance=True)

# Fit the model on training data
# SOLUTION: X_train, y_train
qda_model.fit(X_train, y_train)

# Make predictions on test data
# SOLUTION: X_test
y_pred_qda = qda_model.predict(X_test)

# Calculate accuracy
# SOLUTION: y_test, y_pred_qda
accuracy_qda = accuracy_score(y_test, y_pred_qda)
print(f"QDA Test Accuracy: {accuracy_qda:.3f}")

**Expected Output**: Accuracy around 0.80-0.82

**Interesting**: QDA might be slightly lower than LDA! Why? With limited data, the extra flexibility can hurt (overfitting). We'll explore this more in Task 4.

### Task 2.2: Compare Covariance Matrices

**WHY**: This shows the key difference between LDA and QDA.

**WHAT**: QDA learns one covariance matrix per class (3 total for Iris), while LDA has only one shared matrix.

**HOW**: Look at the values - they're different! Each class has a unique shape.

In [ ]:
# Access covariance matrices (one per class)
qda_covs = qda_model.covariance_
print("Number of covariance matrices in QDA:", len(qda_covs))
print()

print("QDA Covariance Matrix for Class 0 (Setosa):")
print(qda_covs[0])
print()

print("QDA Covariance Matrix for Class 1 (Versicolor):")
print(qda_covs[1])
print()

print("QDA Covariance Matrix for Class 2 (Virginica):")
print(qda_covs[2])

**Answers to Questions**:
1. **How many covariance matrices?** 3 (one per class)
2. **Are they identical?** No! Each class has different values.
3. **Why different from LDA?** LDA assumes all classes have the same spread/shape. QDA allows each class to have its own shape.

**Key Insight**: More parameters to estimate = more flexibility = need more data!

### Task 2.3: Visualize QDA Decision Boundaries

**WHY**: See how QDA's curved boundaries differ from LDA's straight lines.

**WHAT**: Same visualization technique, but now boundaries can curve.

**HOW**: The quadratic terms in the discriminant function create curves.

In [ ]:
# Predict class for each point in the mesh using QDA
Z_qda = qda_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z_qda = Z_qda.reshape(xx.shape)

# Plot
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z_qda, alpha=0.3, cmap='viridis')
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors='k', cmap='viridis')
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title('QDA Decision Boundaries')
plt.show()

**Answers to Questions**:
1. **Straight or curved?** Curved (look carefully at the boundaries)
2. **More flexible?** QDA is more flexible than LDA

**Observation**: The boundaries might be subtly curved. Sometimes the curves are small if the class covariances are similar.

---

## Part 3: LDA vs Logistic Regression (10 minutes)

### Background
**WHY COMPARE?** Both produce linear boundaries, but they approach the problem differently:
- **LDA (generative)**: Models each class distribution, then derives boundaries
- **Logistic Regression (discriminative)**: Models the boundary directly

**KEY QUESTION**: Which approach works better in practice?

### Task 3.1: Fit Logistic Regression

**WHAT**: `multi_class='multinomial'` handles 3+ classes properly.

**HOW**: Logistic regression directly optimizes the classification boundary using gradient descent.

In [ ]:
# Create Logistic Regression model
logreg_model = LogisticRegression(max_iter=200)

# Fit the model
# SOLUTION: X_train, y_train
logreg_model.fit(X_train, y_train)

# Make predictions
# SOLUTION: X_test
y_pred_logreg = logreg_model.predict(X_test)

# Calculate accuracy
accuracy_logreg = accuracy_score(y_test, y_pred_logreg)
print(f"Logistic Regression Test Accuracy: {accuracy_logreg:.3f}")

**Expected Output**: Accuracy around 0.82-0.84

**Observation**: Often similar to LDA! This makes sense - both are linear classifiers.

### Task 3.2: Compare All Three Methods

**WHY**: A summary table makes comparison easy.

**WHAT**: We show accuracy and boundary type for each method.

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(f"{'Model':<25} {'Test Accuracy':>15} {'Boundary Type':>18}")
print("-"*60)
# SOLUTION: accuracy_lda, accuracy_qda
print(f"{'LDA':<25} {accuracy_lda:>15.3f} {'Linear':>18}")
print(f"{'QDA':<25} {accuracy_qda:>15.3f} {'Quadratic':>18}")
print(f"{'Logistic Regression':<25} {accuracy_logreg:>15.3f} {'Linear':>18}")
print("="*60)

**Answers to Questions**:
1. **Best performer?** Usually LDA or Logistic Regression (very close)
2. **Did QDA's flexibility help?** Often NO with small data - the extra parameters can overfit

**Key Takeaway**: More complexity isn't always better! LDA's simplicity can be an advantage with limited data.

### Task 3.3: Visualize All Three Side-by-Side

**WHY**: Visual comparison shows the similarities and differences clearly.

**WHAT**: Three subplots showing all decision boundaries together.

In [ ]:
# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# LDA
Z_lda = lda_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contourf(xx, yy, Z_lda, alpha=0.3, cmap='viridis')
axes[0].scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors='k', cmap='viridis')
axes[0].set_title(f'LDA (Acc: {accuracy_lda:.3f})')
axes[0].set_xlabel(feature_names[0])
axes[0].set_ylabel(feature_names[1])

# QDA
Z_qda = qda_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z_qda, alpha=0.3, cmap='viridis')
axes[1].scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors='k', cmap='viridis')
axes[1].set_title(f'QDA (Acc: {accuracy_qda:.3f})')
axes[1].set_xlabel(feature_names[0])

# Logistic Regression
Z_log = logreg_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[2].contourf(xx, yy, Z_log, alpha=0.3, cmap='viridis')
axes[2].scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors='k', cmap='viridis')
axes[2].set_title(f'Logistic Reg (Acc: {accuracy_logreg:.3f})')
axes[2].set_xlabel(feature_names[0])

plt.tight_layout()
plt.show()

**Answer**: LDA and Logistic Regression boundaries look most similar (both linear).

**Observation**: QDA's boundaries may curve slightly, especially where classes overlap.

---

## Part 4: When to Use Which? (10 minutes)

### Background
**WHY THIS MATTERS**: Choosing the right model for your problem is crucial!

**FACTORS TO CONSIDER**:
- Dataset size
- Class characteristics (similar shapes vs different)
- Presence of outliers/noise
- Computational resources

### Task 4.1: Small Training Dataset Experiment

**WHY**: With limited data, simpler models often win!

**HYPOTHESIS**: QDA will struggle more than LDA with small data because it has more parameters to estimate.

**WHAT**: Reduce training set to only 30 samples and see what happens.

In [ ]:
# Create a smaller training set (only 30 samples)
X_train_small = X_train[:30]
y_train_small = y_train[:30]

print(f"Small training set size: {len(X_train_small)}")

# Train LDA on small dataset
lda_small = LinearDiscriminantAnalysis(store_covariance=True)
lda_small.fit(X_train_small, y_train_small)
acc_lda_small = lda_small.score(X_test, y_test)

# Train QDA on small dataset
# SOLUTION: X_train_small, y_train_small
qda_small = QuadraticDiscriminantAnalysis()
qda_small.fit(X_train_small, y_train_small)
# SOLUTION: X_test, y_test
acc_qda_small = qda_small.score(X_test, y_test)

# Train Logistic Regression on small dataset
logreg_small = LogisticRegression(max_iter=200)
logreg_small.fit(X_train_small, y_train_small)
acc_logreg_small = logreg_small.score(X_test, y_test)

print("\nTest Accuracy with Small Training Set:")
print(f"  LDA: {acc_lda_small:.3f}")
print(f"  QDA: {acc_qda_small:.3f}")
print(f"  Logistic Regression: {acc_logreg_small:.3f}")

**Answers to Questions**:
1. **Worst performer?** Usually QDA (accuracy drops most)
2. **Why QDA struggles?** It needs to estimate 3 separate covariance matrices. With only ~10 samples per class, these estimates are unreliable (high variance).

**Key Insight**: LDA's shared covariance assumption acts as **regularization** - it prevents overfitting with small data!

**Rule of Thumb**: Need at least 10-20 samples per class per estimated parameter. QDA has many more parameters than LDA.

### Task 4.2: Add Noise to Test Robustness

**WHY**: Real data is noisy! We want models that don't break easily.

**WHAT**: Add Gaussian noise to test data and see which model degrades least.

**HYPOTHESIS**: Logistic Regression might be most robust (makes fewer distributional assumptions).

In [ ]:
# Add Gaussian noise to test data
np.random.seed(42)
noise = np.random.normal(0, 0.5, X_test.shape)
X_test_noisy = X_test + noise

# Evaluate all models on noisy data
acc_lda_noisy = lda_model.score(X_test_noisy, y_test)
acc_qda_noisy = qda_model.score(X_test_noisy, y_test)
acc_logreg_noisy = logreg_model.score(X_test_noisy, y_test)

print("\nTest Accuracy on Noisy Data:")
print(f"  LDA: {acc_lda_noisy:.3f}")
print(f"  QDA: {acc_qda_noisy:.3f}")
print(f"  Logistic Regression: {acc_logreg_noisy:.3f}")

print("\nAccuracy Drop:")
print(f"  LDA: {accuracy_lda - acc_lda_noisy:.3f}")
print(f"  QDA: {accuracy_qda - acc_qda_noisy:.3f}")
print(f"  Logistic Regression: {accuracy_logreg - acc_logreg_noisy:.3f}")

**Answers to Questions**:
1. **Smallest drop?** Often Logistic Regression
2. **Is LogReg more robust?** Usually yes - it makes fewer assumptions about data distribution

**Key Insight**: 
- LDA/QDA assume Gaussian distributions - noise violates this
- Logistic Regression just models the boundary - more flexible to non-Gaussian data

**Practical Advice**: If your data has outliers or isn't cleanly Gaussian, prefer Logistic Regression.

---

## Part 5: Reflection Questions (5 minutes)

### Question 1: Equal Covariance Assumption

**Sample Answer**: 
LDA assumes equal covariance because it makes the math simpler and the boundaries linear. When computing the discriminant function, the quadratic term (x^T Σ^-1 x) appears for both classes. If Σ is the same for all classes, this term cancels out when comparing classes, leaving only linear terms. This is why the boundaries are straight lines. If covariances differ (as in QDA), the quadratic terms don't cancel, giving curved boundaries.

### Question 2: When to Use LDA vs QDA

**Sample Answer**: 
Use **LDA** when:
- You have limited data (< 50-100 samples per class)
- Classes seem to have similar spread/shape
- You want a stable, interpretable model
- Computational simplicity matters

Use **QDA** when:
- You have plenty of data (> 100 samples per class)
- Classes clearly have different shapes/spreads
- You need the extra flexibility
- Linear boundaries don't work well

### Question 3: LDA vs Logistic Regression

**Sample Answer**: 
Prefer **LDA** when:
- Data is approximately Gaussian
- You have small datasets
- Classes are well-separated
- You want to understand class distributions

Prefer **Logistic Regression** when:
- Data is not Gaussian or has outliers
- You only care about the boundary (not class distributions)
- Robustness is important
- You want to avoid distributional assumptions

### Question 4: Practical Application

**Sample Answer**: 
With only 50 patients across 3 diseases (~17 per class) and 5 features, I would choose **LDA**. Here's why:

1. **Small dataset**: QDA would need to estimate 3 separate 5×5 covariance matrices (75 parameters), which is too many for ~17 samples per class. LDA only estimates 1 shared covariance (15 parameters).

2. **Medical data**: Blood test measurements are often approximately Gaussian, which suits LDA's assumptions.

3. **Stability**: With limited data, LDA's constraint (equal covariance) acts as regularization, giving more stable predictions.

4. **Interpretability**: Doctors would want to understand what differentiates diseases - LDA's class means are easy to interpret.

I would also compare with Logistic Regression using cross-validation to be sure!

---

## Summary: Core Learning Outcomes ✓

**Congratulations! You've learned to:**

✅ **Use sklearn's LinearDiscriminantAnalysis and QuadraticDiscriminantAnalysis**
   - Import, create, fit, predict, access parameters
   - Standard sklearn API: `fit()`, `predict()`, `score()`

✅ **Explain why LDA assumes equal covariances**
   - Equal Σ → quadratic terms cancel → linear boundaries
   - Different Σ → quadratic terms remain → curved boundaries

✅ **Describe when LDA vs QDA is appropriate**
   - LDA: Small data, similar class shapes, want stability
   - QDA: Large data, different class shapes, need flexibility

✅ **Compare LDA vs logistic regression performance**
   - Both linear, but different estimation methods
   - LDA better for small Gaussian data
   - Logistic more robust to violations

**Key Principles to Remember**:
1. Simpler models often win with limited data
2. Assumptions help when they're met, hurt when violated
3. Always validate empirically - theory doesn't always match practice
4. Understand trade-offs: flexibility vs stability, interpretability vs performance

---

# Part 6: For MPS439 Students Only (30 minutes)

## Advanced Task 6.1: Implement LDA from Scratch

**WHY**: Understanding the implementation deepens your grasp of the algorithm.

**WHAT WE'RE DOING**:
1. Calculate class means (μ_k)
2. Calculate pooled covariance (Σ)
3. Compute discriminant functions
4. Classify based on highest score

**HOW IT WORKS**:
- Discriminant function: δ_k(x) = x^T Σ^(-1) μ_k - (1/2) μ_k^T Σ^(-1) μ_k + log(π_k)
- Pick class k with maximum δ_k(x)

In [ ]:
def lda_from_scratch(X_train, y_train, X_test):
    """
    Implement LDA from scratch.
    
    Parameters:
    - X_train: Training features (n_samples, n_features)
    - y_train: Training labels
    - X_test: Test features (n_samples_test, n_features)
    
    Returns:
    - y_pred: Predicted labels for X_test
    - class_means: Learned class means
    - pooled_cov: Learned pooled covariance
    """
    # Step 1: Calculate class means
    classes = np.unique(y_train)
    n_classes = len(classes)
    n_features = X_train.shape[1]
    
    class_means = np.zeros((n_classes, n_features))
    for i, c in enumerate(classes):
        # SOLUTION: mean(axis=0) computes mean for each feature
        class_means[i] = X_train[y_train == c].mean(axis=0)
    
    # Step 2: Calculate pooled covariance matrix
    # Formula: Σ = (1/(n-K)) * Σ_k (X_k - μ_k)^T (X_k - μ_k)
    pooled_cov = np.zeros((n_features, n_features))
    n_total = len(y_train)
    
    for i, c in enumerate(classes):
        X_c = X_train[y_train == c]
        # Center the data (subtract class mean)
        X_centered = X_c - class_means[i]
        # Add to pooled covariance
        # SOLUTION: X_centered (n_c × p), so X_centered.T @ X_centered gives p × p covariance
        pooled_cov += X_centered.T @ X_centered
    
    # Divide by total number of samples
    pooled_cov = pooled_cov / n_total
    
    # Step 3: Calculate class priors (how common each class is)
    priors = np.zeros(n_classes)
    for i, c in enumerate(classes):
        priors[i] = np.sum(y_train == c) / len(y_train)
    
    # Step 4: Compute discriminant functions for each test sample
    # Inverse of covariance matrix
    cov_inv = np.linalg.inv(pooled_cov)
    
    # Compute discriminant scores
    n_test = X_test.shape[0]
    scores = np.zeros((n_test, n_classes))
    
    for k in range(n_classes):
        # Linear term: x^T * Sigma^-1 * mu_k
        # SOLUTION: multiply with class_means[k]
        linear_term = X_test @ cov_inv @ class_means[k]
        
        # Quadratic term: -0.5 * mu_k^T * Sigma^-1 * mu_k
        quadratic_term = -0.5 * class_means[k] @ cov_inv @ class_means[k]
        
        # Prior term: log(prior_k)
        prior_term = np.log(priors[k])
        
        scores[:, k] = linear_term + quadratic_term + prior_term
    
    # Predict: choose class with highest score
    # SOLUTION: argmax of scores
    y_pred = np.argmax(scores, axis=1)
    
    return y_pred, class_means, pooled_cov

**Code Explanation**:

**Step 1 - Class Means**: 
- For each class, compute the average of all samples in that class
- `mean(axis=0)` averages across samples, keeping one value per feature

**Step 2 - Pooled Covariance**: 
- Center each class by subtracting its mean
- Compute X_centered^T @ X_centered (this is how covariance is calculated)
- Pool (average) across all classes, weighted by degrees of freedom (n - K)

**Step 3 - Priors**: 
- Simply count how often each class appears

**Step 4 - Discriminant Function**: 
- For each class, compute δ_k(x) for all test points
- Linear term: how aligned is x with the class mean (weighted by Σ^-1)
- Quadratic term: normalizes for mean distance from origin
- Prior term: adjusts for class frequency
- Pick class with highest score

## Advanced Task 6.2: Test Your Implementation

**WHY**: Verify that our implementation matches sklearn's professional version.

**WHAT**: Compare predictions and learned parameters.

In [ ]:
# Test your implementation
y_pred_custom, means_custom, cov_custom = lda_from_scratch(X_train, y_train, X_test)

# Calculate accuracy
# SOLUTION: y_test, y_pred_custom
acc_custom = accuracy_score(y_test, y_pred_custom)

# Compare with sklearn's LDA
print(f"Custom LDA Test Accuracy: {acc_custom:.3f}")
print(f"sklearn LDA Test Accuracy: {accuracy_lda:.3f}")
print(f"Difference: {abs(acc_custom - accuracy_lda):.3f}")

# Compare learned parameters
print("\nClass Means Comparison (first class only):")
print(f"Custom:  {means_custom[0]}")
print(f"sklearn: {lda_model.means_[0]}")

print("\nPooled Covariance Comparison (top-left element):")
print(f"Custom:  {cov_custom[0, 0]:.6f}")
print(f"sklearn: {lda_model.covariance_[0, 0]:.6f}")

**Expected Results**:
- Accuracy should be identical or within 0.001
- Class means should match exactly
- Covariance should match exactly

**If there's a difference**: 
- Check your matrix operations
- Make sure you're dividing by (n - K) not n in covariance
- Verify you're centering by the correct class mean

**Key Insight**: sklearn uses the same math we just implemented! The difference is optimization and numerical stability tricks.

## Advanced Task 6.3: Derive Decision Boundary (2-Class Case)

**WHY**: Understanding the math reveals why boundaries are linear.

**MATHEMATICAL DERIVATION**:

Starting with discriminant functions for two classes:

$$\delta_0(x) = x^T\Sigma^{-1}\mu_0 - \frac{1}{2}\mu_0^T\Sigma^{-1}\mu_0 + \log(\pi_0)$$
$$\delta_1(x) = x^T\Sigma^{-1}\mu_1 - \frac{1}{2}\mu_1^T\Sigma^{-1}\mu_1 + \log(\pi_1)$$

Decision boundary occurs where $\delta_0(x) = \delta_1(x)$:

$$x^T\Sigma^{-1}(\mu_0 - \mu_1) = \frac{1}{2}(\mu_0^T\Sigma^{-1}\mu_0 - \mu_1^T\Sigma^{-1}\mu_1) + \log\left(\frac{\pi_1}{\pi_0}\right)$$

This is a **linear equation** in $x$: $w^Tx = b$ where:
- $w = \Sigma^{-1}(\mu_0 - \mu_1)$ (weight vector)
- $b$ is the constant on the right side

**Geometric Interpretation**: The boundary is perpendicular to the line joining the two means, positioned based on priors.

In [ ]:
# Extract data for only classes 0 and 1
mask = (y_train == 0) | (y_train == 1)
X_train_binary = X_train[mask]
y_train_binary = y_train[mask]

mask_test = (y_test == 0) | (y_test == 1)
X_test_binary = X_test[mask_test]
y_test_binary = y_test[mask_test]

# Fit LDA on binary problem
lda_binary = LinearDiscriminantAnalysis()
lda_binary.fit(X_train_binary, y_train_binary)

# Extract parameters
mu_0 = lda_binary.means_[0]
mu_1 = lda_binary.means_[1]
sigma_inv = np.linalg.inv(lda_binary.covariance_)
pi_0 = lda_binary.priors_[0]
pi_1 = lda_binary.priors_[1]

# Calculate decision boundary coefficients
# w = Sigma^-1 * (mu_0 - mu_1)
w = sigma_inv @ (mu_0 - mu_1)

# b = 0.5 * (mu_0^T * Sigma^-1 * mu_0 - mu_1^T * Sigma^-1 * mu_1) + log(pi_1/pi_0)
b = 0.5 * (mu_0 @ sigma_inv @ mu_0 - mu_1 @ sigma_inv @ mu_1) + np.log(pi_1 / pi_0)

print("Decision Boundary Equation: w^T * x = b")
print(f"w = {w}")
print(f"b = {b:.4f}")

# The decision boundary line is: w[0]*x + w[1]*y = b
# Rearranging: y = (b - w[0]*x) / w[1]
print(f"\nDecision line: {w[1]:.4f}*y = {b:.4f} - {w[0]:.4f}*x")
print(f"Simplified: y = {b/w[1]:.4f} - {w[0]/w[1]:.4f}*x")

**Understanding the Output**:

- **w vector**: Direction perpendicular to the decision boundary
- **b value**: Offset that positions the boundary
- **Linear equation**: y = mx + c form shows this is a straight line

**Key Insight**: The equal covariance assumption is what makes this equation linear! If Σ_0 ≠ Σ_1, we'd have quadratic terms that don't cancel.

## Advanced Task 6.4: Visualize the Mathematical Boundary

**WHY**: Seeing the math in action builds intuition.

**WHAT**: Plot the decision line we derived mathematically and compare with sklearn's boundary.

In [ ]:
# Plot the data
plt.figure(figsize=(8, 6))
plt.scatter(X_train_binary[y_train_binary==0, 0], 
            X_train_binary[y_train_binary==0, 1], 
            label='Class 0 (Setosa)', alpha=0.6, s=50)
plt.scatter(X_train_binary[y_train_binary==1, 0], 
            X_train_binary[y_train_binary==1, 1], 
            label='Class 1 (Versicolor)', alpha=0.6, s=50)

# Plot the decision boundary we derived
x_boundary = np.linspace(X_train_binary[:, 0].min() - 0.5, 
                         X_train_binary[:, 0].max() + 0.5, 100)
y_boundary = (b - w[0] * x_boundary) / w[1]
plt.plot(x_boundary, y_boundary, 'r--', linewidth=2, label='Our Derived Boundary')

# Plot class means
plt.scatter([mu_0[0]], [mu_0[1]], c='blue', marker='X', s=200, 
            edgecolors='k', linewidths=2, label='μ₀', zorder=5)
plt.scatter([mu_1[0]], [mu_1[1]], c='orange', marker='X', s=200, 
            edgecolors='k', linewidths=2, label='μ₁', zorder=5)

plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title('LDA Decision Boundary (Mathematically Derived)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Answer**: Yes! The line perfectly matches sklearn's decision boundary.

**Observations**:
1. The boundary is perpendicular to the line connecting the two means
2. If priors are equal, the boundary passes through the midpoint
3. If priors differ, the boundary shifts toward the rarer class (giving it more space)

**Mathematical Beauty**: This single line equation $w^Tx = b$ encapsulates all of LDA's classification logic!

---

## Advanced Summary: What MPS439 Students Learned ✓

**Beyond the core content, you now understand:**

✅ **Derived LDA decision boundaries mathematically**
   - Started from Bayes' theorem
   - Applied Gaussian assumptions
   - Showed why equal covariance → linear boundaries

✅ **Implemented LDA in Python from scratch**
   - Computed class means
   - Calculated pooled covariance matrix
   - Evaluated discriminant functions
   - Matched sklearn's results exactly

✅ **Understood the computational mechanics**
   - Matrix inversions (Σ^-1)
   - Quadratic forms (x^T Σ^-1 μ)
   - How priors affect boundaries

✅ **Connected theory to practice**
   - Visualized mathematical derivations
   - Verified formulas with real data
   - Interpreted geometric meaning

**Key Mathematical Insights**:

1. **Why "Linear"?**: Equal covariance causes quadratic terms to cancel in the discriminant function comparison

2. **Computational Complexity**: 
   - Training: O(n·p²) for covariance + O(p³) for inversion
   - Prediction: O(p²) per sample
   - Where n = samples, p = features

3. **Why Invert Covariance?**: The inverse Σ^-1 appears from the Gaussian density formula. It measures "distance" in feature space, accounting for correlations.

4. **From LDA to QDA**: Simply allow each class to have its own Σ_k. The math stays the same, but quadratic terms no longer cancel.

## Reflection for MPS439 Students

### Question 1: Computational Complexity

**Sample Answer**: 
LDA's complexity is O(n·p²) for training:
- Computing covariance: O(n·p²) (n samples, each contributing to p×p matrix)
- Matrix inversion: O(p³) (usually p << n, so covariance dominates)
- For prediction: O(p²) per sample (matrix-vector products)

This is very efficient compared to methods like SVM or neural networks!

### Question 2: Why Invert Covariance?

**Sample Answer**: 
The inverse appears from the Gaussian density formula:

$$p(x|y=k) = \frac{1}{(2\pi)^{p/2}|\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(x-\mu_k)^T\Sigma^{-1}(x-\mu_k)\right)$$

The quadratic form $(x-\mu_k)^T\Sigma^{-1}(x-\mu_k)$ is the **Mahalanobis distance** - it measures how far x is from μ_k, accounting for the covariance structure. If features are correlated, Σ^-1 adjusts the distance metric appropriately.

Geometrically: Σ^-1 transforms space so that the Gaussian becomes spherical (identity covariance), making distance intuitive.

### Question 3: Modifying to QDA

**Sample Answer**: 
To implement QDA, modify the code to:

1. **Don't pool covariance**: Instead of computing one pooled_cov, compute Σ_k for each class k separately

```python
class_covs = []
for i, c in enumerate(classes):
    X_c = X_train[y_train == c]
    X_centered = X_c - class_means[i]
    cov_k = (X_centered.T @ X_centered) / (len(X_c) - 1)
    class_covs.append(cov_k)
```

2. **Update discriminant function**: Use Σ_k^-1 instead of Σ^-1:

```python
for k in range(n_classes):
    cov_inv_k = np.linalg.inv(class_covs[k])
    # Now the quadratic term x^T Σ_k^-1 x DOESN'T cancel!
    quadratic_term = -0.5 * (X_test @ cov_inv_k @ X_test.T).diagonal()
    linear_term = X_test @ cov_inv_k @ class_means[k]
    constant = -0.5 * class_means[k] @ cov_inv_k @ class_means[k]
    log_det = -0.5 * np.log(np.linalg.det(class_covs[k]))
    scores[:, k] = quadratic_term + linear_term + constant + log_det + np.log(priors[k])
```

The key difference: the quadratic term (x^T Σ_k^-1 x) is now different for each class, creating curved boundaries!

---

**MPS439 Section Complete!** 🎉

You now have deep mathematical and implementation understanding of LDA and QDA. This foundation will help you:
- Understand other discriminant analysis variants
- Recognize when generative vs discriminative approaches are appropriate
- Debug and improve machine learning models
- Read research papers on classification methods

Next week: Decision Trees - a completely non-parametric approach!